# detach-clone-snapshot — faded example 1: Complete the snapshot that records a weight at each step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-clone-snapshot`. Running the beacon reports progress on the `PyTorch: detach + clone snapshot` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: detach + clone snapshot` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-clone-snapshot`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-clone-snapshot"
DD_SUBTOPIC = "PyTorch: detach + clone snapshot"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In an optimizer loop the parameter is updated in place, so appending the live tensor stores aliases that all show the final value. The fix is to append `p.detach().clone()` each step: `detach()` severs the autograd graph and `clone()` allocates fresh storage so the entry is frozen.

## Faded exercise 1

### Complete the per-step weight snapshot

The loop below runs plain gradient descent on a scalar parameter `p` and appends a record of `p` to `traj` each step. Complete the single line that appends a faithful, non-aliasing snapshot so that early entries differ from the final value and none of them track the autograd graph.

**Fill in:** Appends a graph-free, independently-stored copy of the current parameter value to the trajectory list.

In [ ]:
t.manual_seed(0)

def snapshot_weights(target=3.0, lr=0.1, steps=5):
    p = t.tensor([0.0], requires_grad=True)
    traj = []
    for _ in range(steps):
        loss = (p - target) ** 2
        loss.backward()
        with t.no_grad():
            p -= lr * p.grad
            p.grad.zero_()
        snap = p.detach().clone()
        traj.append(snap)
    return t.stack(traj)

result = snapshot_weights()
print(result.squeeze().tolist())

def _test():
    traj = snapshot_weights()
    # ground-truth gradient descent on (p - 3)^2, lr=0.1, start 0
    p = 0.0
    expected = []
    for _ in range(5):
        g = 2 * (p - 3.0)
        p = p - 0.1 * g
        expected.append(p)
    exp = t.tensor(expected).reshape(5, 1)
    assert traj.shape == (5, 1), traj.shape
    assert t.allclose(traj, exp, atol=1e-5), (traj.squeeze().tolist(), expected)
    # no aliasing: entries must differ
    assert float(traj[0]) != float(traj[-1])
    # graph-free snapshots
    assert not traj.requires_grad

try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
t.manual_seed(0)

def snapshot_weights(target=3.0, lr=0.1, steps=5):
    p = t.tensor([0.0], requires_grad=True)
    traj = []
    for _ in range(steps):
        loss = (p - target) ** 2
        loss.backward()
        with t.no_grad():
            p -= lr * p.grad
            p.grad.zero_()
        snap = p.detach().clone()
        traj.append(snap)
    return t.stack(traj)

result = snapshot_weights()
print(result.squeeze().tolist())
```
</details>